# 01 · Corpus exploration

What did we actually collect, and is it good enough to answer clinical questions?

Run `hcrag ingest` first.

In [ ]:
import sys
import warnings

sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_colwidth", 90)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
from pathlib import Path

from src.ingestion.loader import iter_documents

docs = list(iter_documents(Path("../data/raw")))
print(f"{len(docs)} documents loaded")

frame = pd.DataFrame([{
    "document_id": d.document_id,
    "title": d.title,
    "type": d.source_type,
    "pages": d.n_pages,
    "sections": len(d.sections),
    "chars": len(d.full_text),
    "year": d.published,
} for d in docs])
frame.head()

## Composition

In [ ]:
print(frame["type"].value_counts(), "\n")
print(frame[["pages", "sections", "chars"]].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
frame["type"].value_counts().plot.barh(ax=axes[0], color="#4C9BE8", title="Source type")
frame["chars"].plot.hist(bins=40, ax=axes[1], color="#5BB98C", title="Characters per document")
frame["sections"].plot.hist(bins=30, ax=axes[2], color="#8E7CC3", title="Sections per document")
plt.tight_layout()

## Topic coverage\n\nRecall@K is capped by whether the corpus contains the answer at all, so coverage is worth checking before blaming the retriever.

In [ ]:
topics = ["hypertension", "diabetes", "asthma", "heart failure", "atrial fibrillation",
          "stroke", "sepsis", "pneumonia", "kidney disease", "obesity", "osteoporosis",
          "migraine", "COVID", "depression", "anxiety", "anticoagul"]

coverage = pd.DataFrame([{
    "topic": t,
    "documents": int(frame["title"].str.contains(t, case=False, na=False).sum()),
} for t in topics]).sort_values("documents", ascending=False)

coverage.plot.barh(x="topic", y="documents", color="#4C9BE8",
                   title="Documents whose title mentions the topic", legend=False)
plt.tight_layout()
coverage

## Section structure\n\nSection names become citation metadata, so they need to be meaningful rather than boilerplate.

In [ ]:
from collections import Counter

sections = Counter(s for d in docs for s in d.sections)
pd.DataFrame(sections.most_common(20), columns=["section", "count"])

In [ ]:
drug = next(d for d in docs if d.source_type == "drug_label")
print(drug.title, "\n")
for block in drug.blocks[:3]:
    print(f"p{block.page} [{block.section}]  {block.text[:180]}…\n")